In [56]:
#metrics_df["cache_references"].agg(["mean", "median","max", "min"])
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import Markdown, display
from openpyxl import Workbook, load_workbook

metrics_df_new= pd.read_csv("metricas_performance_PC_NOVO.csv", sep=";")
metrics_df_old = pd.read_csv("metricas_performance_PC_ANTIGO.csv", sep=";")
complete_metrics:pd.DataFrame = pd.concat([metrics_df_new, metrics_df_old], ignore_index=True)
collection_margin: pd.DataFrame = complete_metrics[
    [
        "Computer",
        "otimization",
        "matrix_size",
        "cycles_collecting_percentage",
        "instructions_collecting_percentage",
        "branches_collecting_percentage",
        "branch_misses_collecting_percentage",
        "cache_references_collecting_percentage",
        "cache_misses_collecting_percentage",
    ]
]

complete_metrics.drop(columns=["cycles_collecting_percentage","instructions_collecting_percentage",
                                "branches_collecting_percentage","branch_misses_collecting_percentage",
                                "cache_references_collecting_percentage","cache_misses_collecting_percentage"
                                ], inplace=True
                            )

In [57]:
complete_metrics["branch-miss-rate"] = (complete_metrics["branch_misses"] / complete_metrics["branches"]) * 100
complete_metrics["cache-miss-rate"] = (complete_metrics["cache_misses"] / complete_metrics["cache_references"]) * 100
complete_metrics.rename(columns={"Computer": "used_computer", "otimization": "otimization_level"}, inplace=True)
complete_metrics["otimization_level"] = complete_metrics["otimization_level"].replace("00", "O0").replace("03", "O3",).replace("03-native", "O3-native")
complete_metrics["avg_clock"] = round(complete_metrics["avg_clock"] / (10**9), 3)
complete_metrics.rename(columns={"avg_clock":"frequency_ghz"}, inplace=True)



In [58]:
collection_margin.groupby(["Computer", "otimization", "matrix_size"]).agg(["mean", "std"])

cycles_collecting_percentage            \
                                                         mean       std   
Computer otimization matrix_size                                          
antigo   00          1000                              66.070  0.581550   
                     2000                              66.591  0.214240   
                     3000                              66.690  0.190846   
                     4000                              66.623  0.080836   
                     5000                              66.637  0.030569   
         03          1000                              66.426  2.202197   
                     2000                              66.315  0.405223   
                     3000                              66.397  0.279684   
                     4000                              66.555  0.169525   
                     5000                              66.594  0.099242   
         03-native   1000                              65.751  2.048134   
                     2000                              65.888  0.417048   
                     3000                              66.480  0.185592   
                     4000                              66.562  0.128392   
                     5000                              66.587  0.103500   
novo     00          1000                              83.333  0.109955   
                     2000                              83.357  0.140637   
                     3000                              83.300  0.078457   
                     4000                              83.335  0.023214   
                     5000                              83.342  0.018135   
         03          1000                              83.296  0.463398   
                     2000                              83.267  0.083540   
                     3000                              83.302  0.050728   
                     4000                              83.321  0.023310   
                     5000                              83.325  0.011785   
         03-native   1000                              83.389  0.203658   
                     2000                              83.338  0.076129   
                     3000                              83.286  0.039215   
                     4000                              83.319  0.026854   
                     5000                              83.321  0.015239   

                                 instructions_collecting_percentage            \
                                                               mean       std   
Computer otimization matrix_size                                                
antigo   00          1000                                    83.210  0.352609   
                     2000                                    83.340  0.135154   
                     3000                                    83.364  0.118434   
                     4000                                    83.319  0.058585   
                     5000                                    83.307  0.060745   
         03          1000                                    84.193  1.255858   
                     2000                                    83.237  0.240880   
                     3000                                    83.192  0.240222   
                     4000                                    83.294  0.111575   
                     5000                                    83.310  0.062361   
         03-native   1000                                    83.425  1.505259   
                     2000                                    82.996  0.283909   
                     3000                                    83.305  0.086442   
                     4000                                    83.334  0.142844   
                     5000                                    83.329  0.056657   
novo     00          1000                                    83.281  0.081438   
                  

In [59]:

metrics_statistics:pd.DataFrame = complete_metrics.groupby(["used_computer", "otimization_level", "matrix_size"])\
                                    .agg(["mean", "std"]).reset_index().rename(columns={"mean":"media", "std":"desvio_padrao"})
wb = Workbook()
wb.save("metricas_performance.xlsx")


with pd.ExcelWriter("metricas_performance.xlsx", engine="openpyxl", mode="a") as writer:
    complete_metrics.to_excel(writer, sheet_name="tabela_completa", columns=complete_metrics.columns)
    metrics_statistics.to_excel(writer, sheet_name="estatisticas", columns=metrics_statistics.columns)

wb_final = load_workbook("metricas_performance.xlsx")
if "Sheet" in wb_final.sheetnames:
    ws_vazia = wb_final["Sheet"]
    wb_final.remove(ws_vazia)

wb_final.save("metricas_performance.xlsx")

In [60]:
plot_time_to_size = px.scatter(complete_metrics, x="matrix_size", y="time_elapsed",
                               facet_col="used_computer", color="otimization_level", 
                               hover_data=["IPC", "CPI", "branch-miss-rate", "cache-miss-rate"]
                            )
plot_time_to_size.update_layout(xaxis={"type": "category"})
plot_time_to_size.show()
plot_time_to_size.write_html("graficos/scatter_time_to_matrix_size.html")

boxplot_time_to_size = px.box(complete_metrics, x="matrix_size", y="time_elapsed",
                               facet_col="used_computer", color="otimization_level", 
                               hover_data=["IPC", "CPI", "branch-miss-rate", "cache-miss-rate"]
                            )
boxplot_time_to_size.update_layout(xaxis={"type": "category"})
boxplot_time_to_size.show()
boxplot_time_to_size.write_html("graficos/box_time_to_matrix_size.html")


In [61]:
plot_CPI_to_size= px.scatter(complete_metrics, x="matrix_size", y="CPI",
                            facet_col="used_computer", color="otimization_level", 
                            hover_data=["time_elapsed", "IPC", "branch-miss-rate", "cache-miss-rate"],
                        )
plot_CPI_to_size.update_layout(xaxis={"type": "category"})
plot_CPI_to_size.show()
plot_CPI_to_size.write_html("graficos/scatter_CPI_to_matrix_size.html")


boxplot_CPI_to_size= px.box(complete_metrics, x="matrix_size", y="CPI",
                            facet_col="used_computer", color="otimization_level", 
                            hover_data=["time_elapsed", "IPC", "branch-miss-rate", "cache-miss-rate"],
                        )
boxplot_CPI_to_size.update_layout(xaxis={"type": "category"})
boxplot_CPI_to_size.show()
boxplot_CPI_to_size.write_html("graficos/box_CPI_to_matrix_size.html")


In [62]:
plot_IPC_to_size= px.scatter(complete_metrics, x="matrix_size", y="IPC",
                            facet_col="used_computer", color="otimization_level",
                            hover_data=["time_elapsed", "CPI", "branch-miss-rate", "cache-miss-rate"]
                        )
plot_IPC_to_size.update_layout(xaxis={"type": "category"})
plot_IPC_to_size.show()
plot_IPC_to_size.write_html("graficos/scatter_IPC_to_matrix_size.html")

boxplot_IPC_to_size= px.box(complete_metrics, x="matrix_size", y="IPC",
                            facet_col="used_computer", color="otimization_level",
                            hover_data=["time_elapsed", "CPI", "branch-miss-rate", "cache-miss-rate"]
                        )
boxplot_IPC_to_size.update_layout(xaxis={"type": "category"})
boxplot_IPC_to_size.show()
boxplot_IPC_to_size.write_html("graficos/box_IPC_to_matrix_size.html")


In [72]:
plot_branch_miss_rate_to_size = px.scatter(complete_metrics, x="matrix_size", y="branch-miss-rate", facet_col="used_computer", color="otimization_level", hover_data=["time_elapsed", "CPI", "IPC", "cache-miss-rate"])
plot_branch_miss_rate_to_size.update_layout(xaxis={"type": "category"})
plot_branch_miss_rate_to_size.write_html("graficos/scatter_branch_miss_rate_to_matrix_size.html")
plot_branch_miss_rate_to_size.show()


boxplot_branch_miss_rate_to_size = px.box(complete_metrics, x="matrix_size", y="branch-miss-rate", facet_col="used_computer", color="otimization_level", hover_data=["time_elapsed", "CPI", "IPC", "cache-miss-rate"])
boxplot_branch_miss_rate_to_size.update_layout(xaxis={"type": "category"})
boxplot_branch_miss_rate_to_size.write_html("graficos/box_branch_miss_rate_to_matrix_size.html")
boxplot_branch_miss_rate_to_size.show()


In [73]:

plot_cache_miss_rate_to_size = px.scatter(complete_metrics, x="matrix_size", y="cache-miss-rate",
                                        facet_col="used_computer", color="otimization_level",
                                        hover_data=["time_elapsed","CPI", "IPC", "branch-miss-rate"]
                                    )
plot_cache_miss_rate_to_size.update_layout(xaxis={"type": "category"})
plot_cache_miss_rate_to_size.show()
plot_cache_miss_rate_to_size.write_html("graficos/scatter_cache_miss_rate_to_matrix_size.html")

boxplot_cache_miss_rate_to_size = px.box(complete_metrics, x="matrix_size", y="cache-miss-rate",
                                        facet_col="used_computer", color="otimization_level",  
                                        hover_data=["time_elapsed","CPI", "IPC", "branch-miss-rate"]
                                    )
boxplot_cache_miss_rate_to_size.update_layout(xaxis={"type": "category"})
boxplot_cache_miss_rate_to_size.show()
boxplot_cache_miss_rate_to_size.write_html("graficos/box_cache_miss_rate_to_matrix_size.html")

In [74]:
from plotly.subplots import make_subplots

branch_misses_rate_stats = metrics_statistics[["used_computer", "otimization_level", "matrix_size", "branch-miss-rate"]]
plot = make_subplots(rows=1, cols=2, subplot_titles=("PC Antigo", "PC Novo"))


for idx, (used_computer, info) in enumerate(branch_misses_rate_stats.groupby("used_computer"), start=1):
    for matrix_size, data in info.groupby("matrix_size"):
            plot.add_trace(
                go.Bar(
                    name=f"pc {used_computer}-{matrix_size}x{matrix_size}",
                    x=data["otimization_level"],
                    y=data["branch-miss-rate"]["media"],
                    orientation="v",
                    error_y={
                        "type": "data",
                        "array": data["branch-miss-rate"]["desvio_padrao"],
                        "visible": True,
                    },
                ),
                row=1,
                col=idx
            )

plot.update_layout(
    title={"text":"Barras de Erro em relação a Taxa de Erro de Desvio", "x":0.45},
    barmode="group",
    xaxis_title={"text":"Níveis de Otimização", "font":{"size":16, "color":"blue"}},
    yaxis_title={"text":"Média da Taxa de Erro de Desvio", "font":{"size":16, "color":"blue"}}
    )


plot.show()
plot.write_html("graficos/error_bars_branch_miss_rate.html")

In [75]:
cache_misses_rate_stats = metrics_statistics[["used_computer", "otimization_level", "matrix_size", "cache-miss-rate"]]
plot = make_subplots(rows=1, cols=2, subplot_titles=("PC Antigo", "PC Novo"))

for idx, (used_computer, info) in enumerate(cache_misses_rate_stats.groupby("used_computer"), start=1):
    for matrix_size, data in info.groupby("matrix_size"):
        plot.add_trace(
            go.Bar(
                name=f"pc {used_computer}-{matrix_size}x{matrix_size}",
                x=data["otimization_level"],
                y=data["cache-miss-rate"]["media"],
                orientation="v",
                error_y={
                    "type": "data",
                    "array": data["cache-miss-rate"]["desvio_padrao"],
                    "visible": True,
                },
            ),
            row=1,
            col=idx,
        )

plot.update_layout(
    title={"text": "Barras de Erro em relação a Taxa de Erro de Acesso a Cache", "x": 0.45},
    barmode="group",
    xaxis_title={"text": "Níveis de Otimização", "font": {"size": 16, "color": "blue"}},
    yaxis_title={"text": "Média da Erro de Acesso a Cache", "font": {"size": 16, "color": "blue"}},
)


plot.show()
plot.write_html("graficos/error_bars_cache_miss_rate.html")


In [76]:
cpi_stats = metrics_statistics[["used_computer", "otimization_level", "matrix_size", "CPI"]]
plot = make_subplots(rows=1, cols=2, subplot_titles=("PC Antigo", "PC Novo"))

for idx, (used_computer, info) in enumerate(cpi_stats.groupby("used_computer"), start=1):
    for matrix_size, data in info.groupby("matrix_size"):
        plot.add_trace(
            go.Bar(
                name=f"pc {used_computer}-{matrix_size}x{matrix_size}",
                x=data["otimization_level"],
                y=data["CPI"]["media"],
                orientation="v",
                error_y={
                    "type": "data",
                    "array": data["CPI"]["desvio_padrao"],
                    "visible": True,
                },
            ),
            row=1,
            col=idx,
        )

plot.update_layout(
    title={"text": "Barras de Erro em relação a CPI", "x": 0.45},
    barmode="group",
    xaxis_title={"text": "Níveis de Otimização", "font": {"size": 16, "color": "blue"}},
    yaxis_title={"text": "Média da CPI", "font": {"size": 16, "color": "blue"}},
)


plot.show()
plot.write_html("graficos/error_bars_CPI.html")


In [77]:
ipc_stats = metrics_statistics[["used_computer", "otimization_level", "matrix_size", "IPC"]]
plot = make_subplots(rows=1, cols=2, subplot_titles=("PC Antigo", "PC Novo"))

for idx, (used_computer, info) in enumerate(ipc_stats.groupby("used_computer"), start=1):
    for matrix_size, data in info.groupby("matrix_size"):
        plot.add_trace(
            go.Bar(
                name=f"pc {used_computer}-{matrix_size}x{matrix_size}",
                x=data["otimization_level"],
                y=data["IPC"]["media"],
                orientation="v",
                error_y={
                    "type": "data",
                    "array": data["IPC"]["desvio_padrao"],
                    "visible": True,
                },
            ),
            row=1,
            col=idx,
        )

plot.update_layout(
    title={"text": "Barras de Erro em relação a IPC", "x": 0.45},
    barmode="group",
    xaxis_title={"text": "Níveis de Otimização", "font": {"size": 16, "color": "blue"}},
    yaxis_title={"text": "Média da IPC", "font": {"size": 16, "color": "blue"}},
)


plot.show()
plot.write_html("graficos/error_bars_IPC.html")
